# Session 3 — generate eval responses, then score
Runs every arm (base, prompt, and each adapter — plus a 3B base/prompt if v14 trained) on the frozen eval, then computes every metric. ~4h. Needs `data/` **and** `runs/*/adapter` from the training sessions.

The scoring at the bottom also reruns identically on your laptop from `results.zip` — no GPU — which is where you'll write it up.

**Setup (every session):** New Notebook → Accelerator **GPU T4 x2**, Internet
**on**, Add Data → your `refusal-calibration` dataset. Run the pip cell first
(it forces one kernel restart; that's expected), then Run All.

**Carry work between sessions:** Save Version → *Save & Run All* so
`/kaggle/working` persists, **or** download `results.zip`/`probe.zip` at the end
and re-upload them into the dataset. Either way every stage resumes — finished
work is skipped, a killed step continues where it stopped.

In [ ]:
!pip install -q unsloth trl peft datasets transformers accelerate pyyaml

Bootstrap: put the uploaded repo in the working dir and on the path. **Run this before the rest.**

In [ ]:
import shutil, os, sys, glob
dst = '/kaggle/working/repo'
if os.path.exists(os.path.join(dst, 'tests.py')):
    # a good copy already here (e.g. mid-session rerun) — keep it so any trained
    # adapters / generated data under runs/ survive
    print('reusing existing', dst)
else:
    if os.path.exists(dst):
        shutil.rmtree(dst)      # broken/partial copy from an earlier attempt
    hits = glob.glob('/kaggle/input/**/stages.py', recursive=True)
    assert hits, 'stages.py not found under /kaggle/input — add your dataset as Input'
    src = os.path.dirname(hits[0])
    print('copying repo from', src)
    shutil.copytree(src, dst)
os.chdir(dst)
for p in (dst, os.path.join(dst, 'data')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('cwd:', os.getcwd(), '| has tests.py:', os.path.exists('tests.py'))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'tests.py'], check=True)  # logic intact first

In [ ]:
from stages import Ctx, generate, score
ctx = Ctx()
assert ctx.ready, 'no frozen eval — carry data/ over from Session 1'

In [ ]:
generate(ctx)   # GPU, ~10-15 min/arm; resumes a killed arm mid-way

In [ ]:
records = score(ctx)   # CPU — metrics, paired deltas, noise floor, curve

In [ ]:
print('failed stages:', __import__('runner').stage.failed or 'none')
import glob, os
print('arms scored:', sorted(records))
partial = [os.path.basename(os.path.dirname(p)) for p in glob.glob('runs/*/responses.partial')]
print('resumable (unfinished) arms:', partial or 'none')

## Save results

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip data/eval.jsonl data/eval.lock \
    data/labeled.jsonl data/mix_*.jsonl runs/*/responses.jsonl runs/*/responses.partial \
    runs/*/meta.json 2>/dev/null
!cd /kaggle/working/repo && zip -qr /kaggle/working/probe.zip data/probe_raw.jsonl 2>/dev/null
!ls -lh /kaggle/working/*.zip